# Live lab 4 · A completed statistical workflow with a real LLM interpreter

Read after Chapter 14. The LLM translates the natural-language request. A tested tool computes coverage. An explicitly deterministic template reports the tool fields. This is a declared workflow, not a hidden replacement for a failed LLM-written report.

**Live execution.** Restarting and running this notebook performs new local model inference. Saved outputs are one actual development run, not guaranteed answers. No paid API is used. First follow [Python setup](https://chikuang.github.io/teaching/genai/python-workflow.html), then run these commands in a terminal at the cloned course root:

```bash
python -m pip install -r requirements.txt -r requirements-llm.txt
python labs/download_model.py
python -m jupyterlab
```

Open the notebook in JupyterLab. Press **Shift+Enter** in order. Model weights take about 3 GB on disk; CPU float32 inference needs several additional GB of RAM. The first model-loading cell can take time. Do not run it repeatedly while it is busy. Inference remains on your machine.


In [1]:
from pathlib import Path
import os
import sys
import json

start = Path(os.environ.get('GENAI_COURSE_ROOT', Path.cwd())).resolve()
root = next((p for p in [start, *start.parents]
             if (p / 'labs/model.json').is_file()), None)
if root is None:
    raise FileNotFoundError('Open this notebook inside the cloned Teaching-GenAI folder')
sys.path.insert(0, str(root))
from labs.llm_lab import LocalLM
from labs.notebook_support import save_run
print('Course folder:', root.name)
print('Python:', sys.version.split()[0])

Course folder: Teaching-GenAI
Python: 3.12.14


In [2]:
lm = LocalLM(root / '_models/qwen2.5-1.5b')
print(lm.versions)
print('Model loaded; no question has been generated yet.')

/Users/cyeh/Desktop/Teaching-GenAI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python': '3.12.14', 'torch': '2.14.1', 'transformers': '4.57.1', 'platform': 'macOS-27.0.1-arm64-arm-64bit', 'device': 'cpu', 'dtype': 'float32', 'threads': 2}
Model loaded; no question has been generated yet.


## 1. Declare the tasks and acceptance rule before inference

The sample sizes differ from the earlier recorded capstone. These remain authored development cases, not an untouched random benchmark. Keep the intended family and sample size outside the model prompt; they are the scoring labels.

In [3]:
from labs.guided_systems import simulation_workflow
from scipy.stats import t
import numpy as np

tasks = [
    {'question': 'Use iid normal observations with sample size 18 to estimate coverage of the mean plus or minus 1.96 estimated standard errors.', 'family': 'normal', 'n': 18},
    {'question': 'Use iid exponential observations with sample size 45 to estimate coverage of the mean plus or minus 1.96 estimated standard errors.', 'family': 'exponential', 'n': 45},
    {'question': 'Use iid Cauchy observations with sample size 22 to estimate coverage.', 'family': 'unsupported', 'n': 22},
]
print([(t['family'], t['n']) for t in tasks])

[('normal', 18), ('exponential', 45), ('unsupported', 22)]


## 2. Execute and score each stage

`simulation_workflow` performs one real model call per request. It validates fields and limits before invoking the numerical tool. A valid schema may still name the wrong family; compare the interpretation with the intended task separately. Unsupported distributions are refused explicitly.

In [4]:
rows = []
for task in tasks:
    row = simulation_workflow(lm, task['question'])
    row['intent_correct'] = row.get('request') == {'family': task['family'], 'n': task['n']}
    expected_status = 'abstained' if task['family'] == 'unsupported' else 'completed'
    row['task_success'] = row['intent_correct'] and row['status'] == expected_status
    rows.append(row)
    print(row['status'], 'intent correct:', row['intent_correct'])
    print(row.get('report', row.get('error')))
assert len(lm.calls) == len(tasks)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


completed intent correct: True
For iid normal samples with n=18, the approximate z interval's estimated coverage is 0.926 from 500 repetitions. The approximate 95% Monte Carlo interval for that coverage probability is [0.900, 0.946]. The interval procedure's nominal 95% label does not guarantee its actual coverage.


completed intent correct: True
For iid exponential samples with n=45, the approximate z interval's estimated coverage is 0.938 from 500 repetitions. The approximate 95% Monte Carlo interval for that coverage probability is [0.913, 0.956]. The interval procedure's nominal 95% label does not guarantee its actual coverage.


invalid intent correct: False
Unrecognized family


## 3. Check the statistical meaning independently

For normal samples the target is `2*F_t(1.96)-1`, because the procedure uses 1.96 with an estimated standard error. The Monte Carlo interval targets that coverage probability. It is different from the mean interval computed inside each simulated dataset. Five standard errors is a reproducible diagnostic tolerance, not a mathematical guarantee that random checks never fail.

In [5]:
for row in rows:
    if row['status'] != 'completed':
        continue
    result = row['tool_result']
    low, high = result['mc_interval']
    assert 0 <= low <= high <= 1
    assert f"{result['estimate']:.3f}" in row['report']
    if result['family'] == 'normal':
        reference = 2 * t.cdf(1.96, result['n'] - 1) - 1
        mc_se = np.sqrt(reference * (1-reference) / result['repetitions'])
        print('Analytic normal coverage:', round(reference, 4))
        assert abs(result['estimate'] - reference) < 5 * mc_se
print('Descriptive task successes:', sum(r['task_success'] for r in rows), '/', len(rows))
save_run(root, 'capstone-bridge', lm, rows)

Analytic normal coverage: 0.9334
Descriptive task successes: 2 / 3
Saved actual calls and results to capstone-bridge.json


PosixPath('/Users/cyeh/Desktop/Teaching-GenAI/.runs/capstone-bridge.json')

## 4. Add LLM-written reporting only as a separate experiment

The current report is deliberately a Python template. A next experiment may give the same tool JSON to an LLM. Preserve its raw report even if rejected; never silently substitute this template and mark the LLM report successful. Score exact family and sample size, numeric agreement at declared precision, distinction between coverage and confidence level, and absence of universal guarantees. See the failed real reports in Appendix B.

## Practice

1. **Predict:** which error can survive schema validation while producing a perfectly correct calculation for the wrong task?
2. **Implement:** use a scripted normal request for the exponential question. The numerical checks can pass, but `intent_correct` must be false. A Boolean sample size must be rejected before simulation.
3. **Investigate:** freeze a revised prompt before new requests, include unsupported families, and separately report interpretation, execution and narrative scores. A total success count over these three authored cases is descriptive only.